# 94c — Extract and waveform-QC nodal-only candidate gathers (SAFE)

This notebook consumes the **provisional** candidate assignments made by `94b_*`.

It:

1. resolves the existing full three-component gathers;
2. extracts missing recovered candidates from continuous position-coded SDS;
3. selects a consensus waveform reference independently for each provisional shot group;
4. aligns candidates on the vertical component using several common receiver traces;
5. rejects low-similarity events and conservatively suppresses temporal retriggers;
6. writes review CSVs and visual QC figures.

It deliberately **does not stack events and does not modify SQLite**. Its accepted-event
manifest is the review checkpoint before a later stacking notebook.


In [1]:
from pathlib import Path
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, UTCDateTime
from obspy.clients.filesystem.sds import Client as SDSClient
from scipy.signal import correlate, correlation_lags

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
SDS_ROOT = PROJECT_ROOT / 'nodal_sds_position_codes'
INPUT_ROOT = PROJECT_ROOT / '94b_nodal_only_candidate_review'
ASSIGNMENTS_CSV = INPUT_ROOT / '94b_nodal_only_event_assignments_provisional.csv'
GROUPS_CSV = INPUT_ROOT / '94b_nodal_only_shot_groups_provisional.csv'

OUT_ROOT = PROJECT_ROOT / '94c_nodal_only_candidate_waveform_qc'
EXTRACTED_ROOT = OUT_ROOT / 'extracted_all_long_gathers'
FIGURE_ROOT = OUT_ROOT / 'figures'
GROUP_FIGURE_ROOT = FIGURE_ROOT / 'by_group'
for directory in [OUT_ROOT, EXTRACTED_ROOT, FIGURE_ROOT, GROUP_FIGURE_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

# Three-component extraction window relative to the 94b event time.
GATHER_TMIN_S = -0.50
GATHER_TMAX_S = 1.50
COMPONENTS = ['Z', 'N', 'E']
PRIMARY_COMPONENT = 'Z'

# Waveform comparison parameters, closely following notebook 95.
BANDPASS_FREQMIN_HZ = 5.0
BANDPASS_FREQMAX_HZ = 150.0
XCORR_TMIN_S = 0.0
XCORR_TMAX_S = 0.80
MAX_XCORR_SHIFT_S = 0.45
MIN_CORR_COEF = 0.65
MIN_XCORR_TRACES = 3
MIN_XCORR_OVERLAP_FRACTION = 0.45
MAX_RECEIVER_SHIFT_MAD_S = 0.020
REFERENCE_TOP_N_TRACES = 8
REFERENCE_MAX_CANDIDATES = 10

# Only candidates this close in absolute time are automatically treated as duplicate
# triggers. Larger count discrepancies remain visible for human review.
RETRIGGER_GUARD_S = 0.75

MAX_GROUPS = None          # e.g. 3 for a quick test
OVERWRITE_EXTRACTED = False
SHOW_PLOTS = False

print('94b assignments:', ASSIGNMENTS_CSV)
print('Continuous SDS:', SDS_ROOT)
print('SAFE review output:', OUT_ROOT)
print('SQLite will not be opened or modified.')


94b assignments: /Volumes/tachyon/LBSSP_DATA/94b_nodal_only_candidate_review/94b_nodal_only_event_assignments_provisional.csv
Continuous SDS: /Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes
SAFE review output: /Volumes/tachyon/LBSSP_DATA/94c_nodal_only_candidate_waveform_qc
SQLite will not be opened or modified.


## Load and validate the provisional assignments

Existing catalog events normally already have a three-component `DPall` gather.
Recovered events have timestamps but no gather, so those are extracted below.


In [2]:
assignments = pd.read_csv(ASSIGNMENTS_CSV, low_memory=False)
groups_94b = pd.read_csv(GROUPS_CSV, low_memory=False)

assignments['event_time'] = pd.to_datetime(
    assignments['event_time'], utc=True, errors='coerce', format='mixed'
)
assignments['nodal_event_time_utc'] = assignments['event_time'].astype(str)
assignments['assigned_source_x_m'] = pd.to_numeric(
    assignments['assigned_source_x_m'], errors='coerce'
)
assignments['expected_blows'] = pd.to_numeric(assignments['expected_blows'], errors='coerce')

required = ['nodal_event_id', 'group_key', 'event_time', 'candidate_origin', 'assigned_source_x_m']
missing = [column for column in required if column not in assignments.columns]
if missing:
    raise RuntimeError(f'94b assignment file is missing required columns: {missing}')
if assignments[required].isna().any().any():
    display(assignments.loc[assignments[required].isna().any(axis=1), required])
    raise RuntimeError('Required 94b assignment fields contain null values')
if assignments['nodal_event_id'].duplicated().any():
    raise RuntimeError('94b nodal_event_id values are not unique')

print(f'Candidates: {len(assignments):,}')
print(f'Provisional groups: {assignments.group_key.nunique():,}')
display(
    assignments.groupby(['candidate_origin'], dropna=False)
    .size().rename('n_candidates').reset_index()
)


Candidates: 1,148
Provisional groups: 44


,candidate_origin,n_candidates
0,existing_unassigned_catalog,866
1,targeted_continuous_sds,282


## Extract uniformly long three-component gathers

All candidates are re-extracted from continuous position-coded SDS. This avoids the
one-second limit of older `DPall` files and preserves complete waveforms after shifts
as large as ±0.45 s. Each provisional group is loaded as one continuous block and
then sliced into per-event gathers.


> **Reviewed revision:** extracted review gathers are written with a uniform `FLOAT64` MiniSEED encoding. This avoids the mixed-encoding compatibility warnings present in the retained run output. Rerun the extraction cell only if regenerating these review files.


In [8]:
def safe_name(value):
    text = str(value)
    for char in [' ', '/', '\\', ':', ';', ',', '(', ')', '[', ']']:
        text = text.replace(char, '_')
    return text


def sds_location_for_row(row):
    label = str(row.get('nodal_timewindow_label', ''))
    if '_N3_' in label or str(row['group_key']) in {'MAY19_010M', 'MAY19_290M'}:
        return 'N3'
    if '_N1_' in label:
        return 'N1'
    return 'N2'


assignments['sds_location'] = assignments.apply(sds_location_for_row, axis=1)
assignments['resolved_gather_path'] = None
sds = SDSClient(str(SDS_ROOT))
extraction_errors = []

for group_key, block in assignments.groupby('group_key', sort=True):
    locations = sorted(block.sds_location.dropna().unique())
    if len(locations) != 1:
        raise RuntimeError(f'{group_key} spans multiple SDS locations: {locations}')
    location = locations[0]
    start = UTCDateTime(block.event_time.min().to_pydatetime()) + GATHER_TMIN_S - 0.10
    end = UTCDateTime(block.event_time.max().to_pydatetime()) + GATHER_TMAX_S + 0.10
    print(f'Loading long 3C block {group_key}: T1.*.{location}.DP? {start} to {end}')
    continuous = sds.get_waveforms('T1', '*', location, 'DP?', start, end)
    continuous.merge(method=1, fill_value='interpolate')
    if not continuous:
        raise RuntimeError(f'No continuous data returned for {group_key}')

    group_dir = EXTRACTED_ROOT / safe_name(group_key)
    group_dir.mkdir(parents=True, exist_ok=True)
    for index, row in block.iterrows():
        out_path = group_dir / f"{safe_name(row['nodal_event_id'])}_DPall_long.mseed"
        try:
            if OVERWRITE_EXTRACTED or not out_path.exists():
                origin = UTCDateTime(row['event_time'].to_pydatetime())
                gather = continuous.slice(
                    origin + GATHER_TMIN_S, origin + GATHER_TMAX_S
                ).copy()
                if not gather:
                    raise RuntimeError('empty SDS slice')
                gather.write(str(out_path), format='MSEED')#, encoding='FLOAT64')
            assignments.loc[index, 'resolved_gather_path'] = str(out_path)
        except Exception as exc:
            extraction_errors.append({
                'nodal_event_id': row['nodal_event_id'],
                'group_key': group_key,
                'stage': 'extract_long_3c_gather',
                'error': repr(exc),
            })

extraction_errors = pd.DataFrame(
    extraction_errors,
    columns=['nodal_event_id', 'group_key', 'stage', 'error'],
)
assignments['gather_path_exists'] = assignments.resolved_gather_path.map(
    lambda value: bool(value) and Path(str(value)).exists()
)
print('Long 3C gathers:', int(assignments.gather_path_exists.sum()), '/', len(assignments))
print('Extraction errors:', len(extraction_errors))
if len(extraction_errors):
    display(extraction_errors)


Loading long 3C block MAY17_104M: T1.*.N2.DP? 2026-05-17T17:10:01.172000Z to 2026-05-17T17:10:45.760000Z


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will b

Loading long 3C block MAY17_105M: T1.*.N2.DP? 2026-05-17T17:10:46.456000Z to 2026-05-17T17:12:28.366000Z


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will b

Loading long 3C block MAY17_106M: T1.*.N2.DP? 2026-05-17T17:12:28.222000Z to 2026-05-17T17:14:31.008000Z


...

/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will b

Loading long 3C block MAY19_SINKHOLE_110_114M: T1.*.N2.DP? 2026-05-19T13:07:39.382000Z to 2026-05-19T13:09:19.186000Z


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will b

Long 3C gathers: 1148 / 1148
Extraction errors: 0


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


## Waveform comparison helpers

Similarity is measured on several energetic vertical-component receivers, rather than
on one trace. A consensus event is selected as the reference for each group. All three
components are retained and displayed, but the vertical component controls acceptance.


In [9]:
def station_x_m(trace):
    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def preprocess_stream(stream):
    output = Stream()
    for original in stream:
        if not any(str(original.stats.channel).endswith(component) for component in COMPONENTS):
            continue
        trace = original.copy()
        trace.data = trace.data.astype(np.float64)
        trace.detrend('linear')
        trace.taper(max_percentage=0.02, type='hann')
        nyquist = 0.5 / trace.stats.delta
        if BANDPASS_FREQMAX_HZ < 0.95 * nyquist:
            trace.filter(
                'bandpass', freqmin=BANDPASS_FREQMIN_HZ, freqmax=BANDPASS_FREQMAX_HZ,
                corners=4, zerophase=True,
            )
        else:
            trace.filter('highpass', freq=BANDPASS_FREQMIN_HZ, corners=4, zerophase=True)
        trace.stats.receiver_x_m = station_x_m(trace)
        output += trace
    return output


def trace_key(trace):
    return str(trace.stats.station), str(trace.stats.channel)


def stream_by_key(stream):
    return {trace_key(trace): trace for trace in stream}


def relative_series(trace, origin, tmin, tmax, dt=None):
    if dt is None:
        dt = float(trace.stats.delta)
    grid = np.arange(tmin, tmax + 0.5 * dt, dt, dtype=float)
    trace_start = trace.stats.starttime - origin
    trace_time = trace_start + np.arange(trace.stats.npts, dtype=float) * trace.stats.delta
    values = np.interp(grid, trace_time, trace.data.astype(float), left=np.nan, right=np.nan)
    return grid, values


def normalized(values):
    values = np.asarray(values, dtype=float)
    good = np.isfinite(values)
    if good.sum() < 10:
        return None
    result = values - np.nanmedian(values)
    scale = np.nanstd(result)
    if not np.isfinite(scale) or scale <= 0:
        return None
    result[~good] = np.nan
    return result / scale


def overlap_normalized_xcorr(reference, candidate, dt):
    reference = normalized(reference)
    candidate = normalized(candidate)
    if reference is None or candidate is None:
        return np.nan, np.nan, 0
    n_total = min(len(reference), len(candidate))
    reference = reference[:n_total]
    candidate = candidate[:n_total]
    min_overlap = max(20, int(np.ceil(MIN_XCORR_OVERLAP_FRACTION * n_total)))

    ref_mask = np.isfinite(reference).astype(float)
    cand_mask = np.isfinite(candidate).astype(float)
    ref = np.where(np.isfinite(reference), reference, 0.0)
    cand = np.where(np.isfinite(candidate), candidate, 0.0)

    lags = correlation_lags(n_total, n_total, mode='full')
    counts = correlate(cand_mask, ref_mask, mode='full', method='auto')
    dot = correlate(cand, ref, mode='full', method='auto')
    sum_cand = correlate(cand, ref_mask, mode='full', method='auto')
    sum_ref = correlate(cand_mask, ref, mode='full', method='auto')
    sumsq_cand = correlate(cand ** 2, ref_mask, mode='full', method='auto')
    sumsq_ref = correlate(cand_mask, ref ** 2, mode='full', method='auto')

    allowed = (
        (np.abs(lags) <= int(round(MAX_XCORR_SHIFT_S / dt)))
        & (counts >= min_overlap)
    )
    coefficients = np.full(dot.shape, np.nan, dtype=float)
    good = allowed & (counts > 0)
    covariance = dot[good] - sum_cand[good] * sum_ref[good] / counts[good]
    variance_cand = sumsq_cand[good] - sum_cand[good] ** 2 / counts[good]
    variance_ref = sumsq_ref[good] - sum_ref[good] ** 2 / counts[good]
    denominator = np.sqrt(np.maximum(variance_cand, 0) * np.maximum(variance_ref, 0))
    valid = denominator > 0
    good_indices = np.flatnonzero(good)
    coefficients[good_indices[valid]] = covariance[valid] / denominator[valid]
    if not np.isfinite(coefficients).any():
        return np.nan, np.nan, 0
    position = int(np.nanargmax(coefficients))
    return (
        float(-lags[position] * dt),
        float(coefficients[position]),
        int(round(counts[position])),
    )

def select_reference_keys(stream, origin):
    rows = []
    for trace in stream.select(channel=f'*{PRIMARY_COMPONENT}'):
        _, values = relative_series(trace, origin, XCORR_TMIN_S, XCORR_TMAX_S)
        if np.isfinite(values).sum() < 10:
            continue
        centered = values - np.nanmedian(values)
        rows.append({
            'key': trace_key(trace),
            'receiver_x_m': station_x_m(trace),
            'energy': float(np.nansum(centered ** 2)),
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        return [], frame
    frame = frame.sort_values('energy', ascending=False).head(REFERENCE_TOP_N_TRACES)
    return frame['key'].tolist(), frame


def compare_to_reference(reference, reference_origin, candidate, candidate_origin, keys):
    reference_map = stream_by_key(reference)
    candidate_map = stream_by_key(candidate)
    shifts, coefficients, details = [], [], []
    for key in keys:
        if key not in reference_map or key not in candidate_map:
            continue
        reference_trace = reference_map[key]
        candidate_trace = candidate_map[key]
        dt = float(reference_trace.stats.delta)
        if abs(float(candidate_trace.stats.delta) - dt) > 1e-6:
            continue
        _, reference_values = relative_series(
            reference_trace, reference_origin, XCORR_TMIN_S, XCORR_TMAX_S, dt
        )
        _, candidate_values = relative_series(
            candidate_trace, candidate_origin, XCORR_TMIN_S, XCORR_TMAX_S, dt
        )
        shift, coefficient, n_overlap = overlap_normalized_xcorr(
            reference_values, candidate_values, dt
        )
        if np.isfinite(shift) and np.isfinite(coefficient):
            shifts.append(shift)
            coefficients.append(coefficient)
            details.append({
                'station': key[0], 'channel': key[1],
                'shift_s': shift, 'corrcoef': coefficient,
                'n_overlap_samples': n_overlap,
            })
    if not shifts:
        return np.nan, np.nan, 0, np.nan, details
    median_shift = float(np.nanmedian(shifts))
    shift_mad = float(1.4826 * np.nanmedian(np.abs(np.asarray(shifts) - median_shift)))
    return (
        median_shift,
        float(np.nanmedian(coefficients)),
        len(shifts),
        shift_mad,
        details,
    )


def choose_consensus_reference(group, streams, origins):
    # High recovery score is a useful tie-breaker, but consensus similarity decides.
    candidates = group.copy()
    candidates['_recovery_score'] = pd.to_numeric(
        candidates.get('recovery_score'), errors='coerce'
    ).fillna(-np.inf)
    candidates = candidates.sort_values(
        ['_recovery_score', 'event_time'], ascending=[False, True]
    ).head(REFERENCE_MAX_CANDIDATES)
    diagnostics = []
    for _, candidate in candidates.iterrows():
        candidate_id = candidate['nodal_event_id']
        keys, _ = select_reference_keys(streams[candidate_id], origins[candidate_id])
        coefficients = []
        supported = 0
        for other_id, other_stream in streams.items():
            if other_id == candidate_id:
                continue
            shift, coefficient, n_traces, shift_mad, _ = compare_to_reference(
                streams[candidate_id], origins[candidate_id],
                other_stream, origins[other_id], keys,
            )
            if np.isfinite(coefficient) and n_traces >= MIN_XCORR_TRACES:
                coefficients.append(coefficient)
                if (
                    coefficient >= MIN_CORR_COEF
                    and abs(shift) <= MAX_XCORR_SHIFT_S
                    and np.isfinite(shift_mad)
                    and shift_mad <= MAX_RECEIVER_SHIFT_MAD_S
                ):
                    supported += 1
        diagnostics.append({
            'reference_nodal_event_id': candidate_id,
            'n_reference_traces': len(keys),
            'n_supported_other_members': supported,
            'median_corr_to_other_members': (
                float(np.nanmedian(coefficients)) if coefficients else np.nan
            ),
            'recovery_score': candidate.get('recovery_score'),
        })
    diagnostics = pd.DataFrame(diagnostics)
    if diagnostics.empty:
        raise RuntimeError('No usable reference candidates')
    diagnostics['_median'] = diagnostics.median_corr_to_other_members.fillna(-np.inf)
    diagnostics = diagnostics.sort_values(
        ['n_supported_other_members', '_median', 'recovery_score'],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    diagnostics['selected_reference'] = False
    diagnostics.loc[0, 'selected_reference'] = True
    selected = diagnostics.loc[0, 'reference_nodal_event_id']
    return selected, diagnostics.drop(columns=['_median'])


## Visual review helper

Every group receives a four-panel figure. The three upper panels show normalized
receiver-median waveforms for Z, N, and E after the vertical-component alignment.
The lower panels show correlation through time and the final keep/reject decision.


In [10]:
def component_beam(stream, origin, component, shift_s, tmin=-0.05, tmax=0.85):
    traces = list(stream.select(channel=f'*{component}'))
    if not traces:
        return None, None
    dt = float(np.median([trace.stats.delta for trace in traces]))
    grid = np.arange(tmin, tmax + 0.5 * dt, dt)
    arrays = []
    for trace in traces:
        trace_start = trace.stats.starttime - origin
        trace_time = trace_start + np.arange(trace.stats.npts, dtype=float) * trace.stats.delta
        values = np.interp(
            grid - shift_s, trace_time, trace.data.astype(float), left=np.nan, right=np.nan
        )
        values = normalized(values)
        if values is not None:
            arrays.append(values)
    if not arrays:
        return None, None
    return grid, np.nanmedian(np.vstack(arrays), axis=0)


def plot_group_qc(group_key, group_result, streams, origins, output_path):
    ordered = group_result.sort_values('event_time').reset_index(drop=True)
    figure = plt.figure(figsize=(15, 9), constrained_layout=True)
    layout = figure.add_gridspec(2, 3, height_ratios=[1.5, 1.0])
    colors = {
        'accepted_for_future_stack': '#0072B2',
        'rejected_low_waveform_similarity': '#D55E00',
        'rejected_insufficient_common_traces': '#CC79A7',
        'rejected_temporal_retrigger': '#E69F00',
        'rejected_inconsistent_receiver_shifts': '#9467BD',
        'reference': '#000000',
    }

    for column, component in enumerate(COMPONENTS):
        axis = figure.add_subplot(layout[0, column])
        for event_index, row in ordered.iterrows():
            event_id = row['nodal_event_id']
            grid, beam = component_beam(
                streams[event_id], origins[event_id], component,
                float(row['xcorr_shift_s']) if np.isfinite(row['xcorr_shift_s']) else 0.0,
            )
            if beam is None:
                continue
            scale = np.nanpercentile(np.abs(beam), 98)
            if not np.isfinite(scale) or scale <= 0:
                continue
            status = row['waveform_qc_status']
            color = colors.get(status, '#777777')
            alpha = 0.35 if status == 'accepted_for_future_stack' else 0.8
            linewidth = 1.8 if bool(row['is_reference_event']) else 0.7
            if bool(row['is_reference_event']):
                color = colors['reference']
                alpha = 1.0
            axis.plot(grid, beam / scale + event_index, color=color, alpha=alpha, lw=linewidth)
        axis.set(
            title=f'{component} component: aligned receiver-median waveforms',
            xlabel='Time relative to candidate origin (s)',
        )
        if column == 0:
            axis.set_ylabel('Candidate sequence (normalized offset)')
        axis.grid(alpha=0.2)

    axis = figure.add_subplot(layout[1, :2])
    accepted = ordered.accepted_for_future_stack.astype(bool)
    axis.scatter(
        ordered.loc[accepted, 'event_time'], ordered.loc[accepted, 'xcorr_corrcoef'],
        color='#0072B2', s=28, label='accepted',
    )
    axis.scatter(
        ordered.loc[~accepted, 'event_time'], ordered.loc[~accepted, 'xcorr_corrcoef'],
        color='#D55E00', marker='x', s=40, label='rejected',
    )
    axis.axhline(MIN_CORR_COEF, color='black', ls='--', lw=1, label='correlation threshold')
    axis.set(title='Overlap-normalized vertical-component consensus similarity', xlabel='UTC time', ylabel='Median correlation')
    axis.grid(alpha=0.25)
    axis.legend(loc='best')

    axis = figure.add_subplot(layout[1, 2])
    counts = ordered.waveform_qc_status.value_counts()
    bars = axis.barh(
        range(len(counts)), counts.values,
        color=[colors.get(status, '#777777') for status in counts.index],
    )
    axis.set_yticks(range(len(counts)), [status.replace('_', '\n') for status in counts.index])
    axis.bar_label(bars)
    expected = ordered.expected_blows.dropna()
    expected_text = f'{expected.iloc[0]:.0f}' if len(expected) else 'unknown'
    axis.set(
        title=f'Final QC decisions\nexpected field blows: {expected_text}',
        xlabel='Number of candidates',
    )
    axis.grid(axis='x', alpha=0.25)

    source = ordered.assigned_source_x_m.iloc[0]
    figure.suptitle(
        f'{group_key}: source x={source:.0f} m; '
        f'{accepted.sum()}/{len(ordered)} accepted after waveform/retrigger QC',
        fontsize=14,
    )
    figure.savefig(output_path, dpi=170, bbox_inches='tight')
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)


## Run group-by-group waveform QC

Rejections are deliberately conservative:

- insufficient common receiver traces;
- vertical-component median correlation below the threshold;
- a second accepted-looking candidate within the 0.75 s retrigger guard.

Expected field shot counts are reported but are **not** used to force acceptance or
rejection.


In [11]:
member_rows = []
reference_rows = []
processing_errors = []

group_keys = sorted(assignments.group_key.unique())
if MAX_GROUPS is not None:
    group_keys = group_keys[:MAX_GROUPS]

for group_number, group_key in enumerate(group_keys, 1):
    print(f'[{group_number}/{len(group_keys)}] {group_key}')
    group = assignments.loc[
        assignments.group_key.eq(group_key) & assignments.gather_path_exists
    ].copy().sort_values('event_time')
    streams, origins = {}, {}
    try:
        for _, row in group.iterrows():
            event_id = row['nodal_event_id']
            streams[event_id] = preprocess_stream(read(str(row['resolved_gather_path'])))
            origins[event_id] = UTCDateTime(row['event_time'].to_pydatetime())

        reference_id, reference_diagnostics = choose_consensus_reference(group, streams, origins)
        reference_diagnostics.insert(0, 'group_key', group_key)
        reference_rows.extend(reference_diagnostics.to_dict('records'))
        reference_stream = streams[reference_id]
        reference_origin = origins[reference_id]
        reference_keys, reference_key_frame = select_reference_keys(reference_stream, reference_origin)

        group_members = []
        for _, row in group.iterrows():
            event_id = row['nodal_event_id']
            if event_id == reference_id:
                shift, coefficient, n_traces, shift_mad, detail = 0.0, 1.0, len(reference_keys), 0.0, []
            else:
                shift, coefficient, n_traces, shift_mad, detail = compare_to_reference(
                    reference_stream, reference_origin,
                    streams[event_id], origins[event_id], reference_keys,
                )

            if n_traces < MIN_XCORR_TRACES:
                preliminary = False
                status = 'rejected_insufficient_common_traces'
            elif not np.isfinite(coefficient) or coefficient < MIN_CORR_COEF:
                preliminary = False
                status = 'rejected_low_waveform_similarity'
            elif not np.isfinite(shift) or abs(shift) > MAX_XCORR_SHIFT_S:
                preliminary = False
                status = 'rejected_low_waveform_similarity'
            elif not np.isfinite(shift_mad) or shift_mad > MAX_RECEIVER_SHIFT_MAD_S:
                preliminary = False
                status = 'rejected_inconsistent_receiver_shifts'
            else:
                preliminary = True
                status = 'accepted_for_future_stack'

            result = row.to_dict()
            result.update({
                'is_reference_event': event_id == reference_id,
                'reference_nodal_event_id': reference_id,
                'xcorr_shift_s': shift,
                'xcorr_corrcoef': coefficient,
                'xcorr_n_traces': n_traces,
                'xcorr_shift_mad_s': shift_mad,
                'corrected_event_time_utc': (
                    row['event_time'] - pd.to_timedelta(shift, unit='s')
                    if np.isfinite(shift) else pd.NaT
                ),
                'trace_corr_json': json.dumps(detail),
                'preliminary_waveform_accept': preliminary,
                'accepted_for_future_stack': preliminary,
                'waveform_qc_status': status,
                'retrigger_of_nodal_event_id': None,
            })
            group_members.append(result)

        group_result = pd.DataFrame(group_members).sort_values('event_time').reset_index(drop=True)

        # Suppress only close candidates that both passed waveform QC. Keep the stronger
        # consensus match; never force the result to equal the expected field count.
        prelim = group_result.loc[group_result.preliminary_waveform_accept].copy()
        cluster = []
        clusters = []
        for index, row in prelim.sort_values('corrected_event_time_utc').iterrows():
            if not cluster:
                cluster = [index]
                continue
            previous_time = group_result.loc[cluster[-1], 'corrected_event_time_utc']
            if (row['corrected_event_time_utc'] - previous_time).total_seconds() < RETRIGGER_GUARD_S:
                cluster.append(index)
            else:
                clusters.append(cluster)
                cluster = [index]
        if cluster:
            clusters.append(cluster)

        for close_cluster in clusters:
            if len(close_cluster) < 2:
                continue
            winner = max(
                close_cluster,
                key=lambda idx: (
                    float(group_result.loc[idx, 'xcorr_corrcoef']),
                    float(pd.to_numeric(group_result.loc[idx, 'recovery_score'], errors='coerce'))
                    if pd.notna(group_result.loc[idx, 'recovery_score']) else -np.inf,
                ),
            )
            winner_id = group_result.loc[winner, 'nodal_event_id']
            for index in close_cluster:
                if index == winner:
                    continue
                group_result.loc[index, 'accepted_for_future_stack'] = False
                group_result.loc[index, 'waveform_qc_status'] = 'rejected_temporal_retrigger'
                group_result.loc[index, 'retrigger_of_nodal_event_id'] = winner_id

        member_rows.extend(group_result.to_dict('records'))
        plot_group_qc(
            group_key, group_result, streams, origins,
            GROUP_FIGURE_ROOT / f'{safe_name(group_key)}_waveform_qc.png',
        )
    except Exception as exc:
        processing_errors.append({
            'group_key': group_key,
            'stage': 'waveform_qc',
            'error': repr(exc),
            'traceback': traceback.format_exc(),
        })
        print('  ERROR:', repr(exc))

members_qc = pd.DataFrame(member_rows)
references_qc = pd.DataFrame(reference_rows)
processing_errors = pd.DataFrame(
    processing_errors,
    columns=['group_key', 'stage', 'error', 'traceback'],
)

print('QC member rows:', len(members_qc))
print('Group errors:', len(processing_errors))
if len(processing_errors):
    display(processing_errors[['group_key', 'error']])


[1/44] MAY17_104M
[2/44] MAY17_105M
[3/44] MAY17_106M
[4/44] MAY17_107M
[5/44] MAY17_108M
[6/44] MAY17_109M
[7/44] MAY17_110M
[8/44] MAY17_111M
[9/44] MAY17_112M
[10/44] MAY17_113M
[11/44] MAY17_114M
[12/44] MAY17_115M
[13/44] MAY17_116M
[14/44] MAY17_117M
[15/44] MAY17_118M
[16/44] MAY17_119M
[17/44] MAY17_120M
[18/44] MAY17_121M
[19/44] MAY17_122M
[20/44] MAY17_123M
[21/44] MAY17_124M
[22/44] MAY17_125M
[23/44] MAY17_126M
[24/44] MAY17_127M
[25/44] MAY17_128M
[26/44] MAY17_129M
[27/44] MAY17_130M
[28/44] MAY17_131M
[29/44] MAY17_132M
[30/44] MAY17_133M
[31/44] MAY17_134M
[32/44] MAY17_135M
[33/44] MAY17_136M
[34/44] MAY17_137M
[35/44] MAY17_138M
[36/44] MAY17_139M
[37/44] MAY17_140M
[38/44] MAY19_010M
[39/44] MAY19_036M
[40/44] MAY19_122M
[41/44] MAY19_130M
[42/44] MAY19_216M
[43/44] MAY19_290M
[44/44] MAY19_SINKHOLE_110_114M
QC member rows: 1148
Group errors: 0


## Summaries and review exports

In [12]:
if members_qc.empty:
    raise RuntimeError('No group QC results were produced')

summary = (
    members_qc.groupby('group_key', as_index=False)
    .agg(
        assigned_source_x_m=('assigned_source_x_m', 'first'),
        expected_blows=('expected_blows', 'first'),
        n_provisional_candidates=('nodal_event_id', 'size'),
        n_preliminary_waveform_accept=('preliminary_waveform_accept', 'sum'),
        n_accepted_for_future_stack=('accepted_for_future_stack', 'sum'),
        n_rejected_low_similarity=(
            'waveform_qc_status',
            lambda values: int((values == 'rejected_low_waveform_similarity').sum()),
        ),
        n_rejected_insufficient_traces=(
            'waveform_qc_status',
            lambda values: int((values == 'rejected_insufficient_common_traces').sum()),
        ),
        n_rejected_temporal_retrigger=(
            'waveform_qc_status',
            lambda values: int((values == 'rejected_temporal_retrigger').sum()),
        ),
        n_rejected_inconsistent_receiver_shifts=(
            'waveform_qc_status',
            lambda values: int((values == 'rejected_inconsistent_receiver_shifts').sum()),
        ),
        median_corr_all_candidates=(
            'xcorr_corrcoef',
            lambda values: float(np.nanmedian(values)),
        ),
        first_event_utc=('event_time', 'min'),
        last_event_utc=('event_time', 'max'),
    )
)
accepted_corr = (
    members_qc.loc[members_qc.accepted_for_future_stack]
    .groupby('group_key').xcorr_corrcoef.median()
)
summary['median_accepted_corr'] = summary.group_key.map(accepted_corr)
summary['accepted_minus_expected'] = (
    summary.n_accepted_for_future_stack - summary.expected_blows
)
summary['count_review_status'] = np.where(
    summary.expected_blows.isna(),
    'expected_count_unknown',
    np.where(
        summary.accepted_minus_expected.abs() <= np.maximum(2, 0.15 * summary.expected_blows),
        'accepted_count_close_to_field_notes',
        'review_count_difference',
    ),
)

members_qc.to_csv(OUT_ROOT / '94c_nodal_only_candidate_waveform_qc.csv', index=False)
members_qc.loc[members_qc.accepted_for_future_stack].to_csv(
    OUT_ROOT / '94c_nodal_only_candidates_accepted_for_future_stack.csv', index=False
)
members_qc.loc[~members_qc.accepted_for_future_stack].to_csv(
    OUT_ROOT / '94c_nodal_only_candidates_rejected.csv', index=False
)
summary.to_csv(OUT_ROOT / '94c_nodal_only_group_waveform_qc_summary.csv', index=False)
references_qc.to_csv(OUT_ROOT / '94c_consensus_reference_diagnostics.csv', index=False)
extraction_errors.to_csv(OUT_ROOT / '94c_extraction_errors.csv', index=False)
processing_errors.to_csv(OUT_ROOT / '94c_group_processing_errors.csv', index=False)

display(summary)
display(
    members_qc.waveform_qc_status.value_counts(dropna=False)
    .rename_axis('waveform_qc_status').reset_index(name='n_candidates')
)


,group_key,assigned_source_x_m,expected_blows,n_provisional_candidates,n_preliminary_waveform_accept,n_accepted_for_future_stack,n_rejected_low_similarity,n_rejected_insufficient_traces,n_rejected_temporal_retrigger,n_rejected_inconsistent_receiver_shifts,median_corr_all_candidates,first_event_utc,last_event_utc,median_accepted_corr,accepted_minus_expected,count_review_status
0,MAY17_104M,104.0,20,8,7,7,1,0,0,0,0.951963,2026-05-17 17:10:01.772000+00:00,2026-05-17 17:10:44.160000+00:00,0.953017,-13,review_count_difference
1,MAY17_105M,105.0,20,26,20,20,6,0,0,0,0.963675,2026-05-17 17:10:47.056000+00:00,2026-05-17 17:12:26.766000+00:00,0.977250,0,accepted_count_close_to_field_notes
2,MAY17_106M,106.0,20,33,27,27,6,0,0,0,0.729045,2026-05-17 17:12:28.822000+00:00,2026-05-17 17:14:29.408000+00:00,0.746471,7,review_count_difference
3,MAY17_107M,107.0,20,27,21,21,6,0,0,0,0.681702,2026-05-17 17:14:32.390000+00:00,2026-05-17 17:16:24.286000+00:00,0.716448,1,accepted_count_close_to_field_notes
4,MAY17_108M,108.0,20,22,19,19,3,0,0,0,0.950018,2026-05-17 17:16:27.150000+00:00,2026-05-17 17:17:57.718000+00:00,0.954830,-1,accepted_count_close_to_field_notes
5,MAY17_109M,109.0,20,24,21,21,3,0,0,0,0.949747,2026-05-17 17:18:08.994000+00:00,2026-05-17 17:19:57.474000+00:00,0.959315,1,accepted_count_close_to_field_notes
6,MAY17_110M,110.0,20,29,26,26,3,0,0,0,0.919642,2026-05-17 17:20:00.216000+00:00,2026-05-17 17:22:16.948000+00:00,0.928301,6,review_count_difference
7,MAY17_111M,111.0,20,23,16,16,7,0,0,0,0.721278,2026-05-17 17:22:24.314000+00:00,2026-05-17 17:26:04.780000+00:00,0.944691,-4,review_count_difference
8,MAY17_112M,112.0,20,22,22,22,0,0,0,0,0.977234,2026-05-17 17:26:08.560000+00:00,2026-05-17 17:28:07.938000+00:00,0.977234,2,accepted_count_close_to_field_notes
9,MAY17_113M,113.0,20,27,19,19,8,0,0,0,0.820760,2026-05-17 17:28:11.020000+00:00,2026-05-17 17:36:06.322000+00:00,0.941099,-1,accepted_count_close_to_field_notes


,waveform_qc_status,n_candidates
0,accepted_for_future_stack,832
1,rejected_low_waveform_similarity,289
2,rejected_inconsistent_receiver_shifts,26
3,rejected_temporal_retrigger,1


In [13]:
plot_summary = summary.sort_values(['group_key']).reset_index(drop=True)
positions = np.arange(len(plot_summary))
figure, axes = plt.subplots(2, 1, figsize=(17, 10), constrained_layout=True)

axes[0].bar(
    positions, plot_summary.n_provisional_candidates,
    color='#BBBBBB', label='94b provisional candidates',
)
axes[0].bar(
    positions, plot_summary.n_accepted_for_future_stack,
    color='#0072B2', label='accepted after waveform/retrigger QC',
)
axes[0].scatter(
    positions, plot_summary.expected_blows,
    marker='_', s=180, linewidth=3, color='#D55E00', label='field-note expected blows',
)
axes[0].set(title='Nodal-only candidates before and after waveform QC', ylabel='Event count')
axes[0].legend(loc='upper left')
axes[0].grid(axis='y', alpha=0.25)

rejected = (
    plot_summary.n_rejected_low_similarity
    + plot_summary.n_rejected_insufficient_traces
    + plot_summary.n_rejected_temporal_retrigger
    + plot_summary.n_rejected_inconsistent_receiver_shifts
)
axes[1].bar(positions, rejected, color='#D55E00')
axes[1].set(title='Rejected candidate count by provisional group', ylabel='Rejected events')
axes[1].grid(axis='y', alpha=0.25)

for axis in axes:
    axis.set_xticks(positions, plot_summary.group_key, rotation=90)

figure.savefig(FIGURE_ROOT / '94c_all_group_waveform_qc_summary.png', dpi=180, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(figure)

print('Review first:')
print(FIGURE_ROOT / '94c_all_group_waveform_qc_summary.png')
print(GROUP_FIGURE_ROOT)
print()
print('Accepted-event manifest (input to a future stacker):')
print(OUT_ROOT / '94c_nodal_only_candidates_accepted_for_future_stack.csv')
print()
print('SAFE checkpoint: no stacks were created and SQLite was not modified.')


Review first:
/Volumes/tachyon/LBSSP_DATA/94c_nodal_only_candidate_waveform_qc/figures/94c_all_group_waveform_qc_summary.png
/Volumes/tachyon/LBSSP_DATA/94c_nodal_only_candidate_waveform_qc/figures/by_group

Accepted-event manifest (input to a future stacker):
/Volumes/tachyon/LBSSP_DATA/94c_nodal_only_candidate_waveform_qc/94c_nodal_only_candidates_accepted_for_future_stack.csv

SAFE checkpoint: no stacks were created and SQLite was not modified.
